In [1]:

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ast

In [2]:

!wget -q "https://raw.githubusercontent.com/dsrscientist/dataset1/master/tmdb_5000_movies.csv"
!wget -q "https://raw.githubusercontent.com/dsrscientist/dataset1/master/tmdb_5000_credits.csv"

print("Downloaded!")

Downloaded!


In [3]:
from google.colab import files
uploaded = files.upload()


Saving tmdb_5000_movies.csv.zip to tmdb_5000_movies.csv (2).zip
Saving tmdb_5000_credits.csv.zip to tmdb_5000_credits.csv (2).zip


In [5]:
import zipfile

with zipfile.ZipFile('/content/tmdb_5000_movies.csv (1).zip', 'r') as z:
    z.extractall('/content')

with zipfile.ZipFile('/content/tmdb_5000_credits.csv (1).zip', 'r') as z:
    z.extractall('/content')

print("Unzipped successfully!")

Unzipped successfully!


In [6]:
!ls


 sample_data			  tmdb_5000_movies.csv
 tmdb_5000_credits.csv		 'tmdb_5000_movies.csv (1).zip'
'tmdb_5000_credits.csv (1).zip'  'tmdb_5000_movies.csv (2).zip'
'tmdb_5000_credits.csv (2).zip'   tmdb_5000_movies.csv.zip
 tmdb_5000_credits.csv.zip


In [7]:
movies = pd.read_csv('/content/tmdb_5000_movies.csv')
credits = pd.read_csv('/content/tmdb_5000_credits.csv')

print("Movies loaded:", len(movies))
print("Credits loaded:", len(credits))

Movies loaded: 4803
Credits loaded: 4803


In [8]:
credits.columns = ['id', 'title', 'cast', 'crew']
movies = movies.merge(credits, on='title')

print("Total movies after merge:", len(movies))

Total movies after merge: 4809


In [9]:
movies = movies[['id_x', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']].copy()
movies.rename(columns={'id_x': 'movie_id'}, inplace=True)
movies.dropna(inplace=True)

print(movies.shape)

(4806, 7)


In [10]:
import ast

def convert(text):
    return [i['name'] for i in ast.literal_eval(text)]

def convert_cast(text):
    return [i['name'] for i in ast.literal_eval(text)[:3]]

def fetch_director(text):
    return [i['name'] for i in ast.literal_eval(text) if i['job'] == 'Director']

movies['genres']   = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast']     = movies['cast'].apply(convert_cast)
movies['crew']     = movies['crew'].apply(fetch_director)
movies['overview'] = movies['overview'].apply(lambda x: x.split())

print("Done!")

Done!


In [11]:
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']].copy()
movies.dropna(inplace=True)

print(movies.shape)

(4806, 7)


In [13]:
import ast

def convert(text):
    return [i['name'] for i in ast.literal_eval(text)]

def convert_cast(text):
    return [i['name'] for i in ast.literal_eval(text)[:3]]

def fetch_director(text):
    return [i['name'] for i in ast.literal_eval(text) if i['job'] == 'Director']

movies['genres']   = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast']     = movies['cast'].apply(convert_cast)
movies['crew']     = movies['crew'].apply(fetch_director)
movies['overview'] = movies['overview'].apply(lambda x: x.split())

print("Done!")

ValueError: malformed node or string: ['Action', 'Adventure', 'Fantasy', 'Science Fiction']

In [14]:
def convert(text):
    # If already a list, just extract names
    if isinstance(text, list):
        return [i['name'] if isinstance(i, dict) else i for i in text]
    # If it's a string, parse it first
    return [i['name'] for i in ast.literal_eval(text)]

def convert_cast(text):
    if isinstance(text, list):
        return [i['name'] if isinstance(i, dict) else i for i in text[:3]]
    return [i['name'] for i in ast.literal_eval(text)[:3]]

def fetch_director(text):
    if isinstance(text, list):
        return [i['name'] for i in text if isinstance(i, dict) and i.get('job') == 'Director']
    return [i['name'] for i in ast.literal_eval(text) if i['job'] == 'Director']

movies['genres']   = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast']     = movies['cast'].apply(convert_cast)
movies['crew']     = movies['crew'].apply(fetch_director)
movies['overview'] = movies['overview'].apply(lambda x: x.split() if isinstance(x, str) else x)

print("Done!")


Done!


In [15]:
def collapse(L):
    return [i.replace(" ", "") for i in L]

movies['cast']     = movies['cast'].apply(collapse)
movies['crew']     = movies['crew'].apply(collapse)
movies['genres']   = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)

movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x).lower())

new_df = movies[['movie_id', 'title', 'tags']].copy()
new_df.head(3)

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...


In [16]:
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()

print(vectors.shape)  # should print (4806, 5000)

(4806, 5000)


In [17]:
similarity = cosine_similarity(vectors)

print(similarity.shape)  # should print (4806, 4806)

(4806, 4806)


In [18]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

    print(f"\nBecause you liked '{movie}', you might also like:\n")
    for i, score in movies_list:
        print(f"  • {new_df.iloc[i].title}  (similarity: {round(score, 2)})")

In [19]:
recommend("")
recommend("The Dark Knight")
recommend("Interstellar")


Because you liked 'Batman Begins', you might also like:

  • The Dark Knight  (similarity: 0.39)
  • The Dark Knight Rises  (similarity: 0.35)
  • Batman  (similarity: 0.35)
  • Batman & Robin  (similarity: 0.34)
  • Batman  (similarity: 0.33)

Because you liked 'The Dark Knight', you might also like:

  • The Dark Knight Rises  (similarity: 0.42)
  • Batman Begins  (similarity: 0.39)
  • Batman Returns  (similarity: 0.32)
  • Batman Forever  (similarity: 0.29)
  • Batman & Robin  (similarity: 0.29)

Because you liked 'Interstellar', you might also like:

  • Silent Running  (similarity: 0.24)
  • The Martian  (similarity: 0.23)
  • Apollo 13  (similarity: 0.22)
  • Space Pirate Captain Harlock  (similarity: 0.22)
  • Space Cowboys  (similarity: 0.22)


In [20]:
new_df[new_df['title'].str.contains("The Notebook", case=False)]

,movie_id,title,tags
1561,11036,The Notebook,an epic love story centered around an older ma...


In [21]:
recommend("The Notebook")


Because you liked 'The Notebook', you might also like:

  • Lovely, Still  (similarity: 0.31)
  • Veer-Zaara  (similarity: 0.31)
  • The Big Parade  (similarity: 0.29)
  • The Age of Adaline  (similarity: 0.29)
  • Next Stop Wonderland  (similarity: 0.29)
